# 00 — Environment Setup & Credential Configuration

Run this notebook once at the start of every new Colab/Kaggle session.
It installs all dependencies, mounts Google Drive, verifies GPU, and
connects to DagsHub MLflow tracking.

**Before running:**
1. Upload your filled-in `colab_secrets.yaml` to Google Drive at
   `/content/drive/MyDrive/numera-ml/configs/colab_secrets.yaml`
2. Enable GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Install dependencies (run once per session)
import subprocess, sys

packages = [
    # Core ML
    "torch==2.6.0", "torchvision",
    "transformers==4.50.0", "datasets==3.2.0", "evaluate==0.4.3",
    "sentence-transformers==4.0.2", "tokenizers==0.21.0",
    "scikit-learn==1.6.1", "accelerate==1.4.0",
    # MLflow
    "mlflow==2.18.0", "dagshub",
    # XBRL / Data
    "arelle-release", "requests", "httpx==0.28.1",
    "PyMuPDF==1.25.3", "Pillow==11.1.0",
    "pandas==2.2.3", "numpy==2.2.2",
    # Utilities
    "pyyaml", "tqdm", "matplotlib", "seaborn",
    "kaggle",  # Kaggle API client
]

print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ Installation complete")

In [ ]:
# Cell 2 — Mount Google Drive and load secrets
from google.colab import drive
drive.mount('/content/drive')

import yaml
from pathlib import Path

DRIVE_ROOT   = Path("/content/drive/MyDrive/numera-ml")
SECRETS_FILE = DRIVE_ROOT / "configs" / "colab_secrets.yaml"
CONFIG_FILE  = DRIVE_ROOT / "configs" / "training_config.yaml"

if not SECRETS_FILE.exists():
    raise FileNotFoundError(
        f"Secrets file not found: {SECRETS_FILE}\n"
        "Upload colab_secrets.yaml to Google Drive first."
    )

secrets = yaml.safe_load(SECRETS_FILE.read_text())
config  = yaml.safe_load(CONFIG_FILE.read_text())

print("✅ Secrets loaded")
print(f"   MLflow URI : {secrets['mlflow']['tracking_uri']}")
print(f"   EDGAR UA   : {secrets['edgar']['user_agent']}")

In [ ]:
# Cell 3 — Create project directories on Drive
DIRS = [
    DRIVE_ROOT / "data" / "edgar" / "raw_pdfs",
    DRIVE_ROOT / "data" / "edgar" / "xbrl",
    DRIVE_ROOT / "data" / "lse" / "raw_pdfs",
    DRIVE_ROOT / "data" / "gcc" / "raw_pdfs",
    DRIVE_ROOT / "data" / "processed" / "ocr_results",
    DRIVE_ROOT / "data" / "processed" / "table_extractions",
    DRIVE_ROOT / "data" / "annotations",
    DRIVE_ROOT / "data" / "taxonomy",
    DRIVE_ROOT / "models" / "checkpoints" / "layoutlm",
    DRIVE_ROOT / "models" / "checkpoints" / "sbert",
    DRIVE_ROOT / "models" / "exported" / "layoutlm_zone_v1",
    DRIVE_ROOT / "models" / "exported" / "sbert_ifrs_v1",
    DRIVE_ROOT / "experiments",
]

for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ {len(DIRS)} directories ready on Drive")

In [ ]:
# Cell 4 — Verify GPU
import torch

if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️  No GPU detected. Training will be slow on CPU.")
    print("   Go to Runtime → Change runtime type → T4 GPU")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device: {DEVICE}")

In [ ]:
# Cell 5 — Configure MLflow → DagsHub
import os, mlflow

os.environ["MLFLOW_TRACKING_URI"]      = secrets["mlflow"]["tracking_uri"]
os.environ["MLFLOW_TRACKING_USERNAME"] = secrets["mlflow"]["tracking_username"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets["mlflow"]["tracking_password"]

mlflow.set_tracking_uri(secrets["mlflow"]["tracking_uri"])

# Smoke-test: list experiments
try:
    exps = mlflow.search_experiments()
    print(f"✅ MLflow connected — {len(exps)} experiments")
    for e in exps:
        print(f"   [{e.experiment_id}] {e.name}")
except Exception as err:
    print(f"❌ MLflow connection failed: {err}")
    print("   Check your DagsHub credentials in colab_secrets.yaml")

In [ ]:
# Cell 6 — Configure Kaggle API (optional — only needed for Kaggle datasets)
import json

kaggle_cfg = secrets.get("kaggle", {})
if kaggle_cfg.get("username") and kaggle_cfg.get("key"):
    kaggle_dir = Path("/root/.config/kaggle")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(
        json.dumps({"username": kaggle_cfg["username"], "key": kaggle_cfg["key"]})
    )
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print(f"✅ Kaggle configured for user: {kaggle_cfg['username']}")
else:
    print("ℹ️  Kaggle credentials not set — Kaggle API disabled")

In [ ]:
# Cell 7 — HuggingFace login (needed if downloading gated models)
hf_token = secrets.get("huggingface", {}).get("token", "")
if hf_token and hf_token != "<your-hf-token>":
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace logged in")
else:
    print("ℹ️  No HuggingFace token — public models only (sufficient for Phase 4)")

In [ ]:
# Cell 8 — Environment summary
import platform, torch

print("="*55)
print("  Numera ML — Environment Ready")
print("="*55)
print(f"  Python  : {platform.python_version()}")
print(f"  Torch   : {torch.__version__}")
print(f"  Device  : {DEVICE}")
print(f"  Drive   : {DRIVE_ROOT}")
print(f"  MLflow  : {secrets['mlflow']['tracking_uri']}")
print("="*55)
print("  Next: Run 01_edgar_data_collection.ipynb")